# CA-DRL Reproduction Notebook

This notebook reproduces the CA-DRL (Channel Allocation via Deep Reinforcement Learning) results using the simulation and agents implemented in this repository.

It will:

- Build the satellite IoT environment exactly as described in the CA-DRL paper
- Train a DQN agent (CA-DRL)
- Track average delay, throughput, and cumulative reward
- Save intermediate results under `results/ca-drl/`
- Plot training curves and summary statistics for analysis

> Prerequisite: install dependencies from `requirements.txt` and run this notebook from the project root (`ca-ddqn`).

In [ ]:
import os
import json
from typing import List

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Ensure the project root is on sys.path
import sys
sys.path.append(os.getcwd())

from utils.config_loader import ExperimentConfig
from utils.reproducibility import ensure_reproducible_experiment
from environment.wireless_env import SatelliteEnvironment
from algorithms.cadrl_agent import CADRLAgent, CADRLTrainer
from evaluation.metrics import EpisodeMetricsTracker, summarise_algorithm

In [ ]:
# Load configuration and set seeds
cfg = ExperimentConfig.from_yaml()
seed_used = ensure_reproducible_experiment(cfg.random_seed)
seed_used

In [ ]:
# Build environment and CA-DRL agent/trainer
device = "cpu"  # change to "cuda" if you have a GPU and CUDA installed

env = SatelliteEnvironment(cfg)
agent = CADRLAgent(env, cfg.training, device=device)
trainer = CADRLTrainer(env, agent)

env.state_dim, env.action_dim

In [ ]:
# Quick sanity check: run a single episode without training updates
state = env.reset()
done = False
steps = 0
while not done and steps < 10:
    action = np.random.randint(0, env.action_dim)
    next_state, reward, done, info = env.step(action)
    steps += 1
info, steps

In [ ]:
# Train CA-DRL for a configurable number of episodes (interactive run)
num_episodes = 1000  # for full reproduction, you can increase towards cfg.training.num_episodes

tracker = EpisodeMetricsTracker(window_size=cfg.evaluation.metrics_window)
episode_rewards: List[float] = []
episode_delays: List[float] = []

for ep in tqdm(range(num_episodes), desc="Training CA-DRL (notebook)"):
    metrics = trainer.train_episode()
    r = metrics["episode_reward"]
    d = metrics["average_delay"]
    thr = metrics["throughput"]
    tracker.update(r, d, thr)
    episode_rewards.append(r)
    episode_delays.append(d)

len(episode_rewards), np.mean(episode_rewards[-100:]) if len(episode_rewards) >= 100 else np.mean(episode_rewards)

In [ ]:
# Plot training curves: reward and delay vs episode
episodes = np.arange(len(episode_rewards))

plt.figure(figsize=(6, 4))
plt.plot(episodes, episode_rewards, label="Episode reward")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(episodes, episode_delays, label="Average delay", color="orange")
plt.xlabel("Episode")
plt.ylabel("Average delay")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summarise CA-DRL performance from this run
summary = summarise_algorithm("CA-DRL (notebook)", tracker.as_dict())
summary

In [ ]:
# Optionally save notebook run results to a JSON file
out_dir = os.path.join("results", "ca-drl")
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "ca_drl_notebook_run.json")
with open(out_path, "w") as f:
    json.dump({"summary": summary, "num_episodes": num_episodes}, f, indent=2)

out_path